##Zadanie3

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from pyspark.sql.functions import row_number, rank, dense_rank, lag, lead, sum, avg, min, max


In [0]:
df1 = spark.createDataFrame([
( 1, '2011-01-01', 500),
( 1, '2011-01-15', 50),
( 1, '2011-01-22', 250),
( 1, '2011-01-24', 75),
( 1, '2011-01-26', 125),
( 1, '2011-01-28', 175),
( 2, '2011-01-01', 500),
( 2, '2011-01-15', 50),
( 2, '2011-01-22', 25),
( 2, '2011-01-23', 125),
( 2, '2011-01-26', 200),
( 2, '2011-01-29', 250),
( 3, '2011-01-01', 500),
( 3, '2011-01-15', 50 ),
( 3, '2011-01-22', 5000),
( 3, '2011-01-25', 550),
( 3, '2011-01-27', 95 ),
( 3, '2011-01-30', 2500)], ["AccountId", "TranDate", "TranAmt"])

df2 = spark.createDataFrame([
(1,'George', 800),
(2,'Sam', 950),
(3,'Diane', 1100),
(4,'Nicholas', 1250),
(5,'Samuel', 1250),
(6,'Patricia', 1300),
(7,'Brian', 1500),
(8,'Thomas', 1600),
(9,'Fran', 2450),
(10,'Debbie', 2850),
(11,'Mark', 2975),
(12,'James', 3000),
(13,'Cynthia', 3000),
(14,'Christopher', 5000)], ["RowID", "FName", "Salary"])

In [0]:
partition = Window.partitionBy("AccountId").orderBy("TranDate")

df1.withColumn("TotalAmount", F.sum("TranAmt").over(partition))\
   .orderBy("AccountId", "TranDate").show()

+---------+----------+-------+-----------+
|AccountId|  TranDate|TranAmt|TotalAmount|
+---------+----------+-------+-----------+
|        1|2011-01-01|    500|        500|
|        1|2011-01-15|     50|        550|
|        1|2011-01-22|    250|        800|
|        1|2011-01-24|     75|        875|
|        1|2011-01-26|    125|       1000|
|        1|2011-01-28|    175|       1175|
|        2|2011-01-01|    500|        500|
|        2|2011-01-15|     50|        550|
|        2|2011-01-22|     25|        575|
|        2|2011-01-23|    125|        700|
|        2|2011-01-26|    200|        900|
|        2|2011-01-29|    250|       1150|
|        3|2011-01-01|    500|        500|
|        3|2011-01-15|     50|        550|
|        3|2011-01-22|   5000|       5550|
|        3|2011-01-25|    550|       6100|
|        3|2011-01-27|     95|       6195|
|        3|2011-01-30|   2500|       8695|
+---------+----------+-------+-----------+



In [0]:
df1.withColumn("RunningAverage", F.mean("TranAmt").over(partition))\
    .withColumn("TransactionCount", F.count("*").over(partition))\
    .withColumn("MinimumAmount", F.min("TranAmt").over(partition))\
    .withColumn("MaximumAmount", F.max("TranAmt").over(partition))\
    .withColumn("CumulativeTotal", F.sum("TranAmt").over(partition))\
    .orderBy(["AccountId", "TranDate"])\
    .show()


+---------+----------+-------+------------------+----------------+-------------+-------------+---------------+
|AccountId|  TranDate|TranAmt|    RunningAverage|TransactionCount|MinimumAmount|MaximumAmount|CumulativeTotal|
+---------+----------+-------+------------------+----------------+-------------+-------------+---------------+
|        1|2011-01-01|    500|             500.0|               1|          500|          500|            500|
|        1|2011-01-15|     50|             275.0|               2|           50|          500|            550|
|        1|2011-01-22|    250| 266.6666666666667|               3|           50|          500|            800|
|        1|2011-01-24|     75|            218.75|               4|           50|          500|            875|
|        1|2011-01-26|    125|             200.0|               5|           50|          500|           1000|
|        1|2011-01-28|    175|195.83333333333334|               6|           50|          500|           1175|
|

In [0]:
partition2 = Window.partitionBy("AccountId").orderBy("TranDate").rowsBetween(-2, 0)

partition3 = Window.partitionBy("AccountId").orderBy("TranDate")

df_sliding = df1.withColumn("SlideAvg", F.avg("TranAmt").over(partition2))\
    .withColumn("SlideQty", F.count("*").over(partition2))\
    .withColumn("SlideMin", F.min("TranAmt").over(partition2))\
    .withColumn("SlideMax", F.max("TranAmt").over(partition2))\
    .withColumn("SlideTotal", F.sum("TranAmt").over(partition2))\
    .withColumn("RN", F.row_number().over(partition3))\
    .orderBy("AccountId", "TranDate", "RN")

df_sliding.show()


+---------+----------+-------+------------------+--------+--------+--------+----------+---+
|AccountId|  TranDate|TranAmt|          SlideAvg|SlideQty|SlideMin|SlideMax|SlideTotal| RN|
+---------+----------+-------+------------------+--------+--------+--------+----------+---+
|        1|2011-01-01|    500|             500.0|       1|     500|     500|       500|  1|
|        1|2011-01-15|     50|             275.0|       2|      50|     500|       550|  2|
|        1|2011-01-22|    250| 266.6666666666667|       3|      50|     500|       800|  3|
|        1|2011-01-24|     75|             125.0|       3|      50|     250|       375|  4|
|        1|2011-01-26|    125|             150.0|       3|      75|     250|       450|  5|
|        1|2011-01-28|    175|             125.0|       3|      75|     175|       375|  6|
|        2|2011-01-01|    500|             500.0|       1|     500|     500|       500|  1|
|        2|2011-01-15|     50|             275.0|       2|      50|     500|    

In [0]:
partition4 = Window.orderBy("Salary").rowsBetween(Window.unboundedPreceding, Window.currentRow)
partition5 = Window.orderBy("Salary").rangeBetween(Window.unboundedPreceding, Window.currentRow)

df_logical = df2.withColumn("SumByRows", F.sum("Salary").over(partition4))\
                .withColumn("SumByRange", F.sum("Salary").over(partition5))\
                .orderBy("RowID")

df_logical.show()

+-----+-----------+------+---------+----------+
|RowID|      FName|Salary|SumByRows|SumByRange|
+-----+-----------+------+---------+----------+
|    1|     George|   800|      800|       800|
|    2|        Sam|   950|     1750|      1750|
|    3|      Diane|  1100|     2850|      2850|
|    4|   Nicholas|  1250|     4100|      5350|
|    5|     Samuel|  1250|     5350|      5350|
|    6|   Patricia|  1300|     6650|      6650|
|    7|      Brian|  1500|     8150|      8150|
|    8|     Thomas|  1600|     9750|      9750|
|    9|       Fran|  2450|    12200|     12200|
|   10|     Debbie|  2850|    15050|     15050|
|   11|       Mark|  2975|    18025|     18025|
|   12|      James|  3000|    21025|     24025|
|   13|    Cynthia|  3000|    24025|     24025|
|   14|Christopher|  5000|    29025|     29025|
+-----+-----------+------+---------+----------+



##Zadanie 4

In [0]:
salary_window = Window.orderBy("Salary")

df2_windowed = df2.withColumn("NextSalary", F.lead("Salary", 1).over(salary_window))\
    .withColumn("PrevSalary", F.lag("Salary", 1).over(salary_window))\
    .withColumn("FirstSalary", F.first("Salary").over(salary_window))\
    .withColumn("LastSalary", F.last("Salary").over(salary_window))\
    .withColumn("RowNumber", F.row_number().over(salary_window))\
    .orderBy("RowID")

df2_windowed.show()

+-----+-----------+------+----------+----------+-----------+----------+---------+
|RowID|      FName|Salary|NextSalary|PrevSalary|FirstSalary|LastSalary|RowNumber|
+-----+-----------+------+----------+----------+-----------+----------+---------+
|    1|     George|   800|       950|      null|        800|       800|        1|
|    2|        Sam|   950|      1100|       800|        800|       950|        2|
|    3|      Diane|  1100|      1250|       950|        800|      1100|        3|
|    4|   Nicholas|  1250|      1250|      1100|        800|      1250|        4|
|    5|     Samuel|  1250|      1300|      1250|        800|      1250|        5|
|    6|   Patricia|  1300|      1500|      1250|        800|      1300|        6|
|    7|      Brian|  1500|      1600|      1300|        800|      1500|        7|
|    8|     Thomas|  1600|      2450|      1500|        800|      1600|        8|
|    9|       Fran|  2450|      2850|      1600|        800|      2450|        9|
|   10|     Debb